In [42]:
import subprocess
subprocess.run(['pip', 'install', 'groq'], capture_output=True)
print("Done")

Done


In [64]:
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_KEY')

In [43]:
from groq import Groq
GROQ_KEY = "Your_Key"

client = Groq(api_key = GROQ_KEY)

#TESTING
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages = [
    {
        "role":"user",
        "content": "Say exactly: DeliriumWatch agent is online"
    
        }
    ]
)
print(response.choices[0].message.content)

*static noise*

"DeliriumWatch agent is online... all frequencies clear, tracking protocols engaged... awaiting mission briefing..."


In [44]:
import os 
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

In [45]:
import pandas as pd

df = pd.read_csv('deliriumwatch_clean_tv.csv')
year_groups = pd.read_csv('year_groups.csv')

print('Data LOADED')

Data LOADED


In [46]:

X = df.drop(columns=['delirium_label'])
y = df['delirium_label']

train_years = ['2008 - 2010', '2011 - 2013', '2014 - 2016']
test_years  = ['2017 - 2019', '2020 - 2022']
train_mask = year_groups['anchor_year_group'].isin(train_years)
test_mask  = year_groups['anchor_year_group'].isin(test_years)

X_train = X[train_mask.values]
X_test  = X[test_mask.values]
y_train = y[train_mask.values]
y_test  = y[test_mask.values]

print(f"Train: {X_train.shape[0]}")
print(f"Test:  {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

Train: 16040
Test:  9803
Features: 27


In [47]:
# Load pre trained model
import joblib

ensemble = joblib.load('deliriumwatch_ensemble.pkl')
feature_names = joblib.load('feature_names.pkl')
print("Model loaded.")
print(f"Features: {len(feature_names)}")

Model loaded.
Features: 27


In [48]:
# AGENT 1 Risk Assessment Agent
def assess_risk(patient_features, model, feature_names):

    patient_df = pd.DataFrame([patient_features])[feature_names]
    
    # Get probability from ensemble
    risk_score = model.predict_proba(patient_df)[0][1]
    
    if risk_score < 0.30:
        risk_level = "LOW"
        risk_emoji = "🟢"
    elif risk_score < 0.60:
        risk_level = "MEDIUM"
        risk_emoji = "🟡"
    else:
        risk_level = "HIGH"
        risk_emoji = "🔴"
    
    return {
        'risk_score': round(float(risk_score), 4),
        'risk_percentage': f"{risk_score*100:.1f}%",
        'risk_level': risk_level,
        'risk_emoji': risk_emoji
    }



In [49]:
# Test on first patient from test set
test_patient = X_test.iloc[0].to_dict()
result = assess_risk(test_patient, ensemble, list(X.columns))

print("=" * 35)
print("STEP 1  RISK ASSESSMENT")
print("=" * 35)
print(f"Risk Score:  {result['risk_score']}")
print(f"Risk Level:  {result['risk_emoji']} {result['risk_level']}")
print(f"Percentage:  {result['risk_percentage']}")
print(f"Actual label: {'DELIRIUM' if y_test.iloc[0]==1 else 'NO DELIRIUM'}")

STEP 1  RISK ASSESSMENT
Risk Score:  0.2907
Risk Level:  🟢 LOW
Percentage:  29.1%
Actual label: NO DELIRIUM


In [50]:

import shap
#  Explainability Agent
def explain_patient(patient_features, model, feature_names, top_n=5):
    """
    Uses SHAP to explain why a patient has their risk score.
    Returns top features driving the prediction up or down.
    """
    patient_df = pd.DataFrame([patient_features])[feature_names]
    
    # Use XGBoost from ensemble for SHAP
    xgb_model = model.named_estimators_['xgb']
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(patient_df)
    
    # Create feature-SHAP pairs
    shap_df = pd.DataFrame({
        'feature': feature_names,
        'shap_value': shap_values[0],
        'feature_value': patient_df.iloc[0].values
    })
    
    # Sort by absolute SHAP value
    shap_df['abs_shap'] = shap_df['shap_value'].abs()
    shap_df = shap_df.sort_values('abs_shap', ascending=False)
    
    # Split into risk-increasing and risk-decreasing
    increasing = shap_df[shap_df['shap_value'] > 0].head(top_n)
    decreasing = shap_df[shap_df['shap_value'] < 0].head(top_n)
    
    return {
        'top_risk_factors': increasing[['feature', 'shap_value', 'feature_value']].to_dict('records'),
        'top_protective_factors': decreasing[['feature', 'shap_value', 'feature_value']].to_dict('records'),
        'all_shap': shap_df
    }



In [51]:
# Test on same patient
explanation = explain_patient(test_patient, ensemble, list(X.columns))

print("=" * 85)
print("STEP 2 — EXPLAINABILITY")
print("=" * 85)
print("\nTop factors INCREASING risk:")
for f in explanation['top_risk_factors'][:3]:
    print(f"  {f['feature']:<35} SHAP: +{f['shap_value']:.4f}  Value: {f['feature_value']:.3f}")
print("=" * 85)
print("\nTop factors DECREASING risk:")
for f in explanation['top_protective_factors'][:3]:
    print(f"  {f['feature']:<35} SHAP: {f['shap_value']:.4f}  Value: {f['feature_value']:.3f}")
print("=" * 85)    

STEP 2 — EXPLAINABILITY

Top factors INCREASING risk:
  spo2_mean                           SHAP: +0.3309  Value: 99.260
  fentanyl_total                      SHAP: +0.2660  Value: 75.000
  unique_sedation_drugs               SHAP: +0.1272  Value: 1.000

Top factors DECREASING risk:
  avg_sofa                            SHAP: -1.1166  Value: 0.800
  nid_score                           SHAP: -0.2755  Value: 0.378
  propofol_total                      SHAP: -0.2521  Value: 0.000


In [52]:

shap_explanation = explain_patient(
    test_patient, ensemble, list(X.columns)
)


shap_explanation['increasing_risk'] = [
    (r['feature'], r['shap_value'], r['feature_value'])
    for r in shap_explanation['top_risk_factors']
]

In [53]:
# Intervention Planning Agent
def plan_interventions(shap_explanation):
    # Features that nursing staff can actually change
    MODIFIABLE_ACTIONS = {
        'nid_score': 'Consolidate nighttime nursing tasks — '
                     'reduce patient disturbances between 22:00-06:00',
        'propofol_total': 'Review sedation protocol — '
                          'consider dose reduction or alternative',
        'midazolam_total': 'Review benzodiazepine use — '
                           'consider non-benzo alternatives (dexmedetomidine avoided)',
        'lorazepam_total': 'Review benzodiazepine use — '
                           'consider non-benzo alternatives',
        'fentanyl_total': 'Review opioid dosing — '
                          'minimize if pain control allows',
        'rr_mean': 'Assess respiratory support — '
                   'review ventilator settings or oxygen delivery',
        'spo2_min': 'Optimize oxygen delivery — '
                    'check for desaturation events and triggers',
        'nibp_mean': 'Review hemodynamic support — '
                     'assess vasopressor need and fluid balance',
        'unique_sedation_drugs': 'Review polypharmacy — '
                                 'simplify sedation regimen where possible'
    }

    # fEATURES that cant be modified
    NON_MODIFIABLE = [
        'age', 'gender', 'avg_sofa', 'admission_sofa',
        'abp_measured', 'hr_mean', 'hr_std', 'hr_max',
        'spo2_mean', 'rr_std', 'abp_mean'
    ]

    interventions = []
    non_modifiable_risks = []

    for feature, shap_val, value in shap_explanation['increasing_risk']:
        if feature in MODIFIABLE_ACTIONS:
            interventions.append({
                'feature': feature,
                'shap_value': shap_val,
                'current_value': value,
                'action': MODIFIABLE_ACTIONS[feature]
            })
        elif feature in NON_MODIFIABLE:
            non_modifiable_risks.append({
                'feature': feature,
                'shap_value': shap_val,
                'note': 'Cannot be modified — monitor only'
            })

    # Sort by SHAP value — highest impact first
    interventions.sort(key=lambda x: x['shap_value'], reverse=True)

    return {
        'actionable_interventions': interventions,
        'non_modifiable_risks': non_modifiable_risks,
        'total_actionable': len(interventions)
    }




In [54]:
interventions = plan_interventions(shap_explanation)

if interventions['total_actionable'] == 0:
    print("No actionable interventions identified.")
    print("Risk driven by non-modifiable factors.")
else:
    print(f"Found {interventions['total_actionable']} actionable intervention(s):\n")
    for i, item in enumerate(interventions['actionable_interventions'], 1):
        print(f"{i}. {item['feature'].upper()}")
        print(f"   Current value: {item['current_value']:.3f}")
        print(f"   SHAP impact:   +{item['shap_value']:.4f}")
        print(f"   Action: {item['action']}")
        print()

if interventions['non_modifiable_risks']:
    print("Non-modifiable risk factors (monitor only):")
    for item in interventions['non_modifiable_risks']:
        print(f"  - {item['feature']}: {item['note']}")

Found 2 actionable intervention(s):

1. FENTANYL_TOTAL
   Current value: 75.000
   SHAP impact:   +0.2660
   Action: Review opioid dosing — minimize if pain control allows

2. UNIQUE_SEDATION_DRUGS
   Current value: 1.000
   SHAP impact:   +0.1272
   Action: Review polypharmacy — simplify sedation regimen where possible

Non-modifiable risk factors (monitor only):
  - spo2_mean: Cannot be modified — monitor only
  - hr_std: Cannot be modified — monitor only


In [56]:
test_patient = X_test.iloc[0].to_dict()
risk_result = assess_risk(test_patient, ensemble, list(X.columns))
print("risk_result ready:", risk_result['risk_level'])

risk_result ready: LOW


In [57]:
shap_explanation = explain_patient(
    test_patient, ensemble, list(X.columns)
)

# Bridge
shap_explanation['increasing_risk'] = [
    (r['feature'], r['shap_value'], r['feature_value'])
    for r in shap_explanation['top_risk_factors']
]

interventions = plan_interventions(shap_explanation)
print("interventions ready:", interventions['total_actionable'], "found")

interventions ready: 2 found


In [62]:
# STEP 4 — Alert Generation Agent (Groq LLM)
def generate_alert(risk_result, shap_explanation, interventions, patient_id=0):
    """
    Uses Groq LLM to generate a structured clinical alert
    based on risk assessment, SHAP explanation, and interventions.
    """
    
    # Build context for the LLM
    risk_factors_text = "\n".join([
        f"- {r['feature']}: value={r['feature_value']:.2f}, "
        f"SHAP impact=+{r['shap_value']:.4f}"
        for r in shap_explanation['top_risk_factors']
    ])
    
    protective_text = "\n".join([
        f"- {r['feature']}: value={r['feature_value']:.2f}, "
        f"SHAP impact={r['shap_value']:.4f}"
        for r in shap_explanation['top_protective_factors']
    ])
    
    if interventions['total_actionable'] > 0:
        intervention_text = "\n".join([
            f"- {item['feature']}: {item['action']}"
            for item in interventions['actionable_interventions']
        ])
    else:
        intervention_text = "No actionable interventions identified."
    
    # Construct the prompt
    prompt = f"""You are a clinical decision support AI for ICU nurses.
    
Generate a concise, professional clinical alert for the following patient assessment.
Write in plain clinical language. Be specific and actionable. Maximum 150 words.

PATIENT ID: {patient_id}
DELIRIUM RISK SCORE: {risk_result['risk_percentage']} ({risk_result['risk_level']} RISK)

TOP RISK FACTORS (driving risk UP):
{risk_factors_text}

PROTECTIVE FACTORS (driving risk DOWN):
{protective_text}

RECOMMENDED NURSING INTERVENTIONS:
{intervention_text}

Write a clinical alert with these sections:
1. RISK SUMMARY (1 sentence)
2. KEY CONCERNS (2-3 bullet points)  
3. RECOMMENDED ACTIONS (numbered list)
4. MONITORING NOTE (1 sentence)

Do not use markdown headers with #. Use plain text headers in CAPS.
"CRITICAL: Only recommend actions explicitly listed in RECOMMENDED NURSING INTERVENTIONS above. Do not suggest ICU transfers, room changes, or any action not in the list."
"""

    # Call Groq API
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a clinical decision support AI. "
                           "Generate clear, concise ICU nursing alerts. "
                           "Always be specific and actionable."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,  # low temperature = more consistent clinical output
        max_tokens=300
    )
    
    alert_text = response.choices[0].message.content
    
    return {
        'patient_id': patient_id,
        'risk_score': risk_result['risk_score'],
        'risk_level': risk_result['risk_level'],
        'alert_text': alert_text
    }
# Test Step 4
alert = generate_alert(
    risk_result, 
    shap_explanation, 
    interventions, 
    patient_id=0
)

print("=" * 40)
print("STEP 4 — CLINICAL ALERT")
print("=" * 40)
print(f"Patient ID: {alert['patient_id']}")
print(f"Risk: {alert['risk_level']} ({alert['risk_score']})")
print()
print(alert['alert_text'])


STEP 4 — CLINICAL ALERT
Patient ID: 0
Risk: LOW (0.2907)

**RISK SUMMARY**
This patient is at low risk for delirium (29.1%) but has several risk factors that need to be addressed.

**KEY CONCERNS**
• The patient's mean SpO2 is 99.26%, which may indicate mild hypoxemia.
• Fentanyl total dose is 75.00, which may be contributing to delirium risk.
• The patient is receiving a single sedation drug, but polypharmacy is still a concern.

**RECOMMENDED ACTIONS**
1. Review opioid dosing and minimize fentanyl total dose if pain control allows.
2. Review polypharmacy and simplify sedation regimen where possible.

**MONITORING NOTE**
Continuously monitor the patient's SpO2 and adjust oxygen therapy as needed.


In [59]:
# MASTER AGENT — connects all 4 steps
def run_delirium_agent(patient_idx, model, X_test, 
                        y_test, feature_names):
    """
    Full agentic pipeline for one patient.
    Runs all 4 steps autonomously.
    """
    print(f"\n{'='*50}")
    print(f"DELIRIUMWATCH CLINICAL ALERT AGENT")
    print(f"Patient Index: {patient_idx}")
    print(f"{'='*50}")
    
    # Extract patient
    patient_features = X_test.iloc[patient_idx].to_dict()
    actual_label = y_test.iloc[patient_idx]
    
    # STEP 1 — Risk Assessment
    print("\n[1/4] Running risk assessment...")
    risk_result = assess_risk(
        patient_features, model, feature_names
    )
    print(f"      Risk: {risk_result['risk_emoji']} "
          f"{risk_result['risk_level']} "
          f"({risk_result['risk_percentage']})")
    
    # STEP 2 — Explainability
    print("[2/4] Computing SHAP explanation...")
    shap_explanation = explain_patient(
        patient_features, model, feature_names
    )
    shap_explanation['increasing_risk'] = [
        (r['feature'], r['shap_value'], r['feature_value'])
        for r in shap_explanation['top_risk_factors']
    ]
    print(f"      Top risk factor: "
          f"{shap_explanation['top_risk_factors'][0]['feature']}")
    
    # STEP 3 — Intervention Planning
    print("[3/4] Planning interventions...")
    interventions = plan_interventions(shap_explanation)
    print(f"      Actionable interventions: "
          f"{interventions['total_actionable']}")
    
    # STEP 4 — Alert Generation
    print("[4/4] Generating clinical alert...")
    alert = generate_alert(
        risk_result, shap_explanation, 
        interventions, patient_idx
    )
    
    # Final output
    print(f"\n{'='*50}")
    print(f"CLINICAL ALERT — PATIENT {patient_idx}")
    print(f"{'='*50}")
    print(f"Risk Level: {risk_result['risk_emoji']} "
          f"{risk_result['risk_level']} "
          f"({risk_result['risk_percentage']})")
    print(f"Actual Outcome: "
          f"{'⚠️ DELIRIUM' if actual_label==1 else '✅ NO DELIRIUM'}")
    print(f"\n{alert['alert_text']}")
    print(f"\n{'='*50}")
    
    return {
        'patient_idx': patient_idx,
        'risk_result': risk_result,
        'shap_explanation': shap_explanation,
        'interventions': interventions,
        'alert': alert,
        'actual_label': actual_label
    }

# Test on 3 different patients
for idx in [0, 5, 10]:
    result = run_delirium_agent(
        idx, ensemble, X_test, 
        y_test, list(X.columns)
    )


DELIRIUMWATCH CLINICAL ALERT AGENT
Patient Index: 0

[1/4] Running risk assessment...
      Risk: 🟢 LOW (29.1%)
[2/4] Computing SHAP explanation...
      Top risk factor: spo2_mean
[3/4] Planning interventions...
      Actionable interventions: 2
[4/4] Generating clinical alert...

CLINICAL ALERT — PATIENT 0
Risk Level: 🟢 LOW (29.1%)
Actual Outcome: ✅ NO DELIRIUM

**RISK SUMMARY**
Patient 0 is at low risk for delirium (29.1%), but multiple risk factors are driving this risk upward.

**KEY CONCERNS**
• The patient's mean SpO2 is slightly below normal (99.26%), which may indicate mild hypoxemia.
• The patient is receiving a high cumulative dose of fentanyl (75.00 mcg), which may contribute to delirium risk.
• The patient is receiving a single sedation medication, but this is still a risk factor for delirium.

**RECOMMENDED ACTIONS**
1. Review the patient's opioid dosing and minimize fentanyl if pain control allows.
2. Simplify the patient's sedation regimen by reducing the number of sed

In [60]:
# Find high risk patients
y_pred_proba = ensemble.predict_proba(X_test)[:, 1]
high_risk_indices = [i for i, p in enumerate(y_pred_proba) if p > 0.70]
print(f"High risk patients found: {len(high_risk_indices)}")
print(f"First 5 indices: {high_risk_indices[:5]}")

High risk patients found: 2269
First 5 indices: [16, 25, 48, 53, 58]


In [61]:
result = run_delirium_agent(
    16, ensemble, X_test, 
    y_test, list(X.columns)
)


DELIRIUMWATCH CLINICAL ALERT AGENT
Patient Index: 16

[1/4] Running risk assessment...
      Risk: 🔴 HIGH (90.8%)
[2/4] Computing SHAP explanation...
      Top risk factor: nid_score
[3/4] Planning interventions...
      Actionable interventions: 2
[4/4] Generating clinical alert...

CLINICAL ALERT — PATIENT 16
Risk Level: 🔴 HIGH (90.8%)
Actual Outcome: ⚠️ DELIRIUM

**RISK SUMMARY**
Patient 16 is at HIGH RISK for delirium with a score of 90.8%.

**KEY CONCERNS**
• Patient's high blood pressure (nibp_mean: 88.58) may indicate the need for vasopressor support or fluid balance review.
• The patient's location in the Coronary Care Unit (CCU) may contribute to the high delirium risk score.
• The patient's low oxygen saturation (spo2_min: 88.00) may be a concern for respiratory status.

**RECOMMENDED ACTIONS**
1. Review hemodynamic support and assess vasopressor need and fluid balance.
2. Consider relocating the patient to a quieter area of the ICU.
3. Consolidate nighttime nursing tasks to